# Détecteur et classifieur d'objets spatiaux et de débris
## Pipeline de classification et de localisation

### Objectif du notebook

Ce notebook prépare le pipeline complet de données avec **Pandas** et **TensorFlow**.
Chaque image contient un **unique objet** (satellite ou débris). Le pipeline génère ainsi deux cibles pour chaque image :
- `class_output` : l'identifiant de la classe (parmi 11 classes)
- `bbox_output` : les 4 coordonnées normalisées `[ymin, xmin, ymax, xmax]`

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tensorflow as tf
import yaml

from UTILS.logging_settings import setup_logging
import logging

# Development: colored console + timestamped file + cleaned/displayed traceback
setup_logging(level=logging.DEBUG, log_file="log_file", console=True, enable_traceback=True)

# Configuration globale
IMG_SIZE = 224
BATCH_SIZE = 16

# Choix du périphérique : GPU s'il est disponible, sinon CPU
GPU_DEVICES = tf.config.list_physical_devices("GPU")
if GPU_DEVICES:
    for gpu in GPU_DEVICES:
        try:
            # Alloue la mémoire GPU au besoin au lieu de tout réserver dès le départ
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            # GPU déjà initialisé (cellule réexécutée) : le réglage reste celui de la première exécution
            pass
    TRAINING_DEVICE = "/GPU:0"
    gpu_names = [tf.config.experimental.get_device_details(gpu).get("device_name", gpu.name) for gpu in GPU_DEVICES]
    logging.info(f"Entraînement sur GPU : {gpu_names}")
else:
    TRAINING_DEVICE = "/CPU:0"
    logging.warning("Aucun GPU détecté : entraînement sur CPU")



DATA_YAML_PATH = 'data/data.yaml'
with open(DATA_YAML_PATH, 'r', encoding='utf-8') as yaml_file:
    dataset_config = yaml.safe_load(yaml_file)

ORIGINAL_CLASS_NAMES = dataset_config['names']
CLASS_NAMES = ['satellite', 'debris']
NUM_CLASSES = len(CLASS_NAMES)

logging.info(f"TensorFlow version: {tf.__version__}")
logging.info(f"Classes originales : {ORIGINAL_CLASS_NAMES}")
logging.info(f"Classes du modèle : {CLASS_NAMES}")

## 1. Préparation des données

### 1.1. Préparation des DataFrames Pandas

Nous lisons les annotations YOLO (`[class_id, x_center, y_center, width, height]`) et les convertissons directement en `[ymin, xmin, ymax, xmax]`.
Chaque image contenant un unique objet, nous n'avons plus besoin de padding `MAX_BOXES`.


In [ ]:
def create_dataset_dataframe(image_dir):
    """
    Lit les annotations YOLO et crée un DataFrame avec une cible binaire.
    Tous les objets sauf debris sont regroupés dans la classe satellite.
    """
    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
    records = []

    for img_path in image_paths:
        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')
        if os.path.exists(label_path):
            with open(label_path, 'r') as label_file:
                line = label_file.readline().strip()
                if line:
                    parts = line.split()
                    cls_id = int(parts[0])
                    x_c, y_c, box_width, box_height = map(float, parts[1:5])
                    original_class_name = (
                        ORIGINAL_CLASS_NAMES[cls_id]
                        if 0 <= cls_id < len(ORIGINAL_CLASS_NAMES)
                        else 'unknown'
                    )

                    ymin = max(0.0, y_c - (box_height / 2.0))
                    xmin = max(0.0, x_c - (box_width / 2.0))
                    ymax = min(1.0, y_c + (box_height / 2.0))
                    xmax = min(1.0, x_c + (box_width / 2.0))

                    binary_class_id = int(original_class_name == 'debris')
                    records.append({
                        'image_path': img_path,
                        'class_id': cls_id,
                        'class_name': original_class_name,
                        'binary_class_id': binary_class_id,
                        'binary_class_name': CLASS_NAMES[binary_class_id],
                        'ymin': ymin,
                        'xmin': xmin,
                        'ymax': ymax,
                        'xmax': xmax,
                    })

    return pd.DataFrame(records)

# Création des DataFrames
df_train = create_dataset_dataframe('data/train/images')
df_val = create_dataset_dataframe('data/valid/images')
df_test = create_dataset_dataframe('data/test/images')

logging.info(f"Train : {len(df_train)} images")
logging.info(f"Val   : {len(df_val)} images")
logging.info(f"Test  : {len(df_test)} images")
logging.info("Distribution binaire (Train) :")
logging.info(df_train['binary_class_name'].value_counts())
df_train.head()

## 2. Exploration des données (EDA)
Vérifions la distribution des classes dans l'ensemble d'entraînement.


In [ ]:
logging.info("Distribution des classes binaires (Train) :")
logging.info(df_train['binary_class_name'].value_counts())

### 2.1. Gestion du déséquilibre des classes

Le jeu d'entraînement est très déséquilibré : environ 10 satellites pour 1 débris.

Plutôt que de construire un jeu 50/50 artificiel (qui jetait des satellites et dupliquait les débris par rotation, en créant des coins noirs présents uniquement sur les débris), nous gardons **toutes les images originales** et compensons le déséquilibre :
- **poids de classe** dans la loss : chaque erreur sur un débris compte environ 10 fois plus qu'une erreur sur un satellite ;
- **augmentation aléatoire** (retournements, rotations) appliquée aux deux classes pendant l'entraînement ;
- **métriques adaptées** : recall et précision sur les débris, et aire sous la courbe précision-rappel (PR-AUC), au lieu de l'accuracy seule.

## 3. Construction du pipeline TensorFlow (`tf.data`)

Nous transformons directement les DataFrames Pandas en `tf.data.Dataset` optimisés via `tf.data.Dataset.from_tensor_slices` :
- Chargement et décodage multithreadés (`AUTOTUNE`)
- Redimensionnement à `IMG_SIZE` et normalisation dans `[0, 1]`
- Sortie au format dictionnaire `{'class_output': ..., 'bbox_output': ...}` prêt pour un modèle multi-tâches (classification + localisation)


In [ ]:
def df_to_tf_dataset(df, img_size=IMG_SIZE, batch_size=BATCH_SIZE, is_training=True):
    image_paths = df['image_path'].values
    bboxes = df[['ymin', 'xmin', 'ymax', 'xmax']].values.astype(np.float32)
    class_ids = df['binary_class_id'].values.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, bboxes, class_ids))

    def load_and_preprocess(img_path, bbox, cls_id):
        img_raw = tf.io.read_file(img_path)
        img = tf.image.decode_jpeg(img_raw, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32) / 255.0
        return img, {
            'class_output': cls_id,
            'bbox_output': bbox,
        }

    if is_training:
        dataset = dataset.shuffle(
            buffer_size=min(len(df), 1000),
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = df_to_tf_dataset(df_train, is_training=True)
val_ds = df_to_tf_dataset(df_val, is_training=False)
test_ds = df_to_tf_dataset(df_test, is_training=False)

logging.info("Datasets TensorFlow construits et prêts pour l'entraînement !")

## 4. Visualisation et vérification des données

### 4.1. Visualisation d'un lot d'entraînement
Affichons un lot d'entraînement avec l'étiquette de classe et la boîte englobante correspondante.


In [ ]:
# for images, targets in train_ds.take(1):
#     fig, axes = plt.subplots(2, 4, figsize=(14, 7))
#     axes = axes.flatten()

#     for i in range(min(8, len(images))):
#         img = images[i].numpy()
#         cls_id = int(targets['class_output'][i].numpy())
#         ymin, xmin, ymax, xmax = targets['bbox_output'][i].numpy()

#         h, w, _ = img.shape
#         box_y = ymin * h
#         box_x = xmin * w
#         box_h = (ymax - ymin) * h
#         box_w = (xmax - xmin) * w

#         axes[i].imshow(img)
#         rect = patches.Rectangle((box_x, box_y), box_w, box_h, linewidth=2, edgecolor='lime', facecolor='none')
#         axes[i].add_patch(rect)
#         axes[i].set_title(f"{CLASS_NAMES[cls_id]}", color='white', backgroundcolor='black', fontsize=10)
#         axes[i].axis('off')

#     plt.tight_layout()
#     plt.show()
#     break


### 4.2. Conversion des images en HSV

In [ ]:
# from pathlib import Path

# import matplotlib.pyplot as plt
# import numpy as np
# from matplotlib.colors import rgb_to_hsv
# from PIL import Image

# images_dir = Path("data/test/images")
# image_paths = sorted(
#      path for path in images_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
# )[:20]
# #
# # Bornes reglables des trois composantes HSV : H (teinte), S (saturation), V (luminosite).
# # Pour les modifier, conserver des valeurs comprises entre 0.0 et 1.0.
# hue_range = (0.0, 1.0)
# saturation_range = (0.0, 1.0)
# value_range = (0.05, 1.0)

# # Reinitialise les resultats pour ne pas reutiliser ceux d'une execution precedente.
# hsv_images = []

# # Traite chaque image individuellement.
# for image_path in image_paths:
#     # Charge l'image en RGB et normalise ses pixels entre 0 et 1.
#     arr = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.float32) / 255.0

#     # Convertit l'image RGB dans l'espace HSV.
#     hsv = rgb_to_hsv(arr)

#     # Construit un masque qui conserve les surfaces sombres du satellite et ses reflets.
#     mask = (
#         (hsv[:, :, 0] >= hue_range[0]) & (hsv[:, :, 0] <= hue_range[1])
#         & (hsv[:, :, 1] >= saturation_range[0]) & (hsv[:, :, 1] <= saturation_range[1])
#         & (hsv[:, :, 2] >= value_range[0]) & (hsv[:, :, 2] <= value_range[1])
#     )

#     # Met a zero les pixels du fond qui ne respectent pas les plages HSV.
#     hsv_filtered = hsv.copy()
#     hsv_filtered[~mask] = 0.0
#     hsv_images.append(hsv_filtered)

# # Affiche les images par pages de 10.
# images_per_page = 10
# for page_start in range(0, len(image_paths), images_per_page):
#     page_paths = image_paths[page_start:page_start + images_per_page]
#     page_hsv = hsv_images[page_start:page_start + images_per_page]
#     figure, axes = plt.subplots(len(page_paths), 2, figsize=(10, 3 * len(page_paths)))
#     axes = np.atleast_2d(axes)

#     # Affiche chaque paire RGB/HSV avec le nom du fichier.
#     for row, (image_path, hsv) in enumerate(zip(page_paths, page_hsv)):
#         arr = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.float32) / 255.0
#         axes[row, 0].imshow(arr)
#         axes[row, 0].set_title(f"RGB - {image_path.name}")
#         axes[row, 1].imshow(hsv)
#         axes[row, 1].set_title("HSV filtre HSV")
#         axes[row, 0].axis("off")
#         axes[row, 1].axis("off")

#     # Ajuste la mise en page, affiche la page et libere la figure.
#     figure.tight_layout()
#     plt.show()
#     plt.close(figure)

# logging.info(f"{len(hsv_images)} images filtrees avec le reglage HSV.")

## 5. Construction du modèle de classification

Construction d'un réseau convolutif qui prédit la probabilité que l'objet soit un `debris` (sortie sigmoïde unique), entraîné sur le jeu original avec des poids de classe.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


def df_to_classification_dataset(
    df,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    is_training=True,
):
    image_paths = df['image_path'].values
    class_ids = df['binary_class_id'].values.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, class_ids))

    def load_and_preprocess(img_path, class_id):
        img_raw = tf.io.read_file(img_path)
        img = tf.image.decode_jpeg(img_raw, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32)
        img = keras.applications.mobilenet_v2.preprocess_input(img)
        return img, class_id

    if is_training:
        dataset = dataset.shuffle(
            buffer_size=min(len(df), 1000),
            reshuffle_each_iteration=True,
        )

    return dataset.map(
        load_and_preprocess,
        num_parallel_calls=tf.data.AUTOTUNE,
    ).batch(batch_size).prefetch(tf.data.AUTOTUNE)


# Entraînement sur le jeu original déséquilibré : le déséquilibre est compensé par les poids de classe.
train_classification_ds = df_to_classification_dataset(df_train, is_training=True)

# Validation sur le jeu complet data/valid, qui garde la proportion réelle de débris.
val_classification_ds = df_to_classification_dataset(df_val, is_training=False)
test_classification_ds = df_to_classification_dataset(df_test, is_training=False)

# Poids de classe n_total / (n_classes * n_classe) : les deux classes pèsent autant dans la loss.
class_counts = df_train['binary_class_id'].value_counts().sort_index()
CLASS_WEIGHT = {
    int(class_id): len(df_train) / (len(class_counts) * count)
    for class_id, count in class_counts.items()
}

logging.info(f"Train : {len(df_train)} images")
logging.info(df_train['binary_class_name'].value_counts())
logging.info(f"Validation : {len(df_val)} images")
logging.info(df_val['binary_class_name'].value_counts())
logging.info("Poids de classe :", {CLASS_NAMES[class_id]: round(weight, 2) for class_id, weight in CLASS_WEIGHT.items()})


from pathlib import Path
model_path = Path("models/Class_Detection/satellite_debris_best.keras")
model = keras.models.load_model(model_path, compile=False)
logging.info(f"Modèle rechargé depuis : {model_path}")


## 6. Suivi visuel des prédictions

### 6.1. Aperçu du résultat à chaque epoch

Le callback affiche une image de validation par classe et la classe prédite.

In [ ]:
class EpochPreviewCallback(keras.callbacks.Callback):
    def __init__(self, dataset, class_names, threshold=0.5):
        super().__init__()
        self.dataset = dataset
        self.class_names = class_names
        # Probabilité de debris à partir de laquelle l'objet est classé debris
        self.threshold = threshold
        self.examples_by_class = {class_id: [] for class_id in range(len(class_names))}

        for images, class_ids in dataset:
            for index in range(images.shape[0]):
                class_id = int(class_ids[index].numpy())
                self.examples_by_class[class_id].append(images[index].numpy())

    def on_epoch_end(self, epoch, logs=None):
        predicted_counts = np.zeros(len(self.class_names), dtype=np.int32)

        for images, _ in self.dataset:
            predictions = self.model.predict(images, verbose=0)
            predicted_class_ids = (predictions[:, 0] >= self.threshold).astype(int)
            predicted_counts += np.bincount(
                predicted_class_ids,
                minlength=len(self.class_names),
            )

        logging.info(
            f"Epoch {epoch + 1} - détections validation : "
            f"{self.class_names[0]}={predicted_counts[0]}, "
            f"{self.class_names[1]}={predicted_counts[1]}"
        )

        for class_id, class_name in enumerate(self.class_names):
            examples = self.examples_by_class[class_id]
            image = examples[np.random.randint(len(examples))][np.newaxis, ...]
            prediction = self.model.predict(image, verbose=0)
            predicted_class_id = int(prediction[0, 0] >= self.threshold)

            figure, axis = plt.subplots(figsize=(5, 5))
            # MobileNetV2 attend des pixels dans [-1, 1]. Pour l'affichage,
            # reconvertit uniquement une copie vers l'intervalle RGB [0, 1].
            display_image = np.clip((image[0] + 1.0) / 2.0, 0.0, 1.0)
            axis.imshow(display_image)
            axis.set_title(
                f"Epoch {epoch + 1} | Réel : {class_name} "
                f"| Prévu : {self.class_names[predicted_class_id]}"
            )
            axis.axis("off")
            figure.tight_layout()
            plt.show()

            if wandb.run is not None:
                wandb.log({
                    f"epoch_preview_{class_name}": wandb.Image(
                        figure,
                        caption=(
                            f"Epoch {epoch + 1} - exemple {class_name} - "
                            f"prévu : {self.class_names[predicted_class_id]}"
                        ),
                    )
                })

            plt.close(figure)

## 7. Suivi de l'expérience avec Weights & Biases

`wandb` enregistre les paramètres, les pertes et les métriques de chaque entraînement. Il permet de comparer plusieurs expériences dans un tableau de bord.

In [ ]:
import wandb
from wandb.integration.keras import WandbMetricsLogger
wandb.init(project="satellite_debris_classification", name="mobilenetv2_balanced")
wandb_callback = WandbMetricsLogger(log_freq="epoch")

## 8. Entraînement du modèle

À chaque epoch, TensorFlow apprend à distinguer `satellite` et `debris` sur le jeu original, avec des poids de classe pour compenser le faible nombre de débris. Le meilleur modèle est choisi sur la PR-AUC de validation, plus fiable que l'accuracy sur des données déséquilibrées. W&B reçoit les métriques grâce au callback `WandbMetricsLogger`.

In [ ]:

import os
from pathlib import Path
from tensorflow import keras

NEXT_MODEL_PATH = Path("models/Class_Detection/satellite_debris_hard_70_retrain_v3.keras")
NEXT_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

LEARNING_RATE_MODE="hardcoded" 
INITIAL_LEARNING_RATE=1e-3

NB_EPOCHS = 70
# Choix possibles : hardcoded, plateau ou cosine_decay.
# Exemple : LEARNING_RATE_MODE=plateau jupyter notebook main.ipynb
LEARNING_RATE_MODE = os.getenv("LEARNING_RATE_MODE", "cosine_decay").strip().lower()
LEARNING_RATE_MODE = {
    "hardcode": "hardcoded",
    "cosine": "cosine_decay",
}.get(LEARNING_RATE_MODE, LEARNING_RATE_MODE)
INITIAL_LEARNING_RATE = float(os.getenv("INITIAL_LEARNING_RATE", "1e-3"))

if LEARNING_RATE_MODE not in {"cosine_decay", "plateau", "hardcoded"}:
    raise ValueError(
        "LEARNING_RATE_MODE doit être 'hardcoded', 'plateau' ou 'cosine_decay'"
    )
try:
    # Construction et entraînement sur le périphérique choisi dans la configuration globale
    with tf.device(TRAINING_DEVICE):
        base_model = keras.applications.MobileNetV2(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            include_top=False,
            weights='imagenet',
        )
        base_model.trainable = True

        # Augmentation aléatoire des deux classes, active uniquement pendant l'entraînement.
        # fill_mode="reflect" évite les coins noirs créés par les rotations.
        data_augmentation = keras.Sequential(
            [
                keras.layers.RandomFlip("horizontal_and_vertical"),
                keras.layers.RandomRotation(0.5, fill_mode="reflect"),
            ],
            name="data_augmentation",
        )

        inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = data_augmentation(inputs)
        x = base_model(x, training=True)
        x = keras.layers.GlobalAveragePooling2D()(x)
        x = keras.layers.Dropout(0.3)(x)
        # Sortie unique : probabilité que l'objet soit un debris
        outputs = keras.layers.Dense(
            1,
            activation="sigmoid",
            name="class_output",
        )(x)
        model = keras.Model(inputs, outputs)

        steps_per_epoch = len(df_train) // BATCH_SIZE
        total_steps = steps_per_epoch * NB_EPOCHS

        if LEARNING_RATE_MODE == "cosine_decay":
            learning_rate = keras.optimizers.schedules.CosineDecay(
                initial_learning_rate=INITIAL_LEARNING_RATE,
                decay_steps=total_steps,
                alpha=1e-2,
            )
        else:
            # Utilisé directement pour le mode hardcoded et comme valeur initiale
            # lorsque ReduceLROnPlateau ajuste le taux pendant l'entraînement.
            learning_rate = INITIAL_LEARNING_RATE

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
            loss=keras.losses.BinaryCrossentropy(),
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Recall(name="debris_recall"),
                keras.metrics.Precision(name="debris_precision"),
                keras.metrics.AUC(curve="PR", name="pr_auc"),
            ],
        )


        checkpoint = keras.callbacks.ModelCheckpoint(
            NEXT_MODEL_PATH,
            monitor="val_pr_auc",
            mode="max",
            save_best_only=True,
        )

        early_stopping = keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=65,
            restore_best_weights=True,
        )

        epoch_preview = EpochPreviewCallback(val_classification_ds, CLASS_NAMES)
        callbacks = [checkpoint, early_stopping, epoch_preview, wandb_callback]

        if LEARNING_RATE_MODE == "plateau":
            callbacks.append(
                keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss",
                    factor=0.5,
                    patience=2,
                    min_lr=1e-6,
                    verbose=1,
                )
            )

        logging.info(
            f"Mode du learning rate : {LEARNING_RATE_MODE} "
            f"(initial={INITIAL_LEARNING_RATE:g})"
        )

        history = model.fit(
            train_classification_ds,
            validation_data=val_classification_ds,
            epochs=NB_EPOCHS,
            class_weight=CLASS_WEIGHT,
            callbacks=callbacks,
        )

    logging.info(f"Modèle ImageNet sauvegardé dans : {NEXT_MODEL_PATH}")
except Exception as e:
    logging.error(f"Erreur pendant l'entraînement : {e}")

## 9. Analyse des résultats

### 9.1. Visualisation des courbes d'apprentissage

Les courbes permettent de comparer les performances sur les données d'entraînement et de validation.

### 9.2. Choix du seuil de décision

Le modèle donne une probabilité de `debris` ; un seuil la transforme en classe. Grâce aux poids de classe, le seuil de 0.5 est un point de départ raisonnable.

Le tableau montre, **sur la validation uniquement**, le compromis entre débris trouvés et fausses alertes pour plusieurs seuils. La validation ne contient que 21 débris : un seuil optimisé finement sur ces quelques images risque de ne pas se généraliser au test. Modifier `DECISION_THRESHOLD` seulement si un écart net apparaît, puis le garder figé pour le test.

In [ ]:
DECISION_THRESHOLD = 0.5

try:
    y_val = df_val['binary_class_id'].values
    p_val = model.predict(val_classification_ds, verbose=0)[:, 0]

    threshold_table = []
    for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        predicted_debris = p_val >= threshold
        threshold_table.append({
            'seuil': threshold,
            'débris trouvés': f"{int((predicted_debris & (y_val == 1)).sum())}/{int((y_val == 1).sum())}",
            'fausses alertes': f"{int((predicted_debris & (y_val == 0)).sum())}/{int((y_val == 0).sum())}",
        })

    logging.info(f"Seuil utilisé : {DECISION_THRESHOLD}")
    pd.DataFrame(threshold_table)
except Exception as e:
    logging.error(f"Erreur pendant l'évaluation du seuil : {e}")


## 10. Évaluation finale sur le jeu de test

Le jeu de test n'est pas utilisé pour modifier le modèle ni pour choisir le seuil. Il sert uniquement à mesurer ses performances sur des images qu'il n'a pas vues pendant l'entraînement.

In [ ]:
from sklearn.metrics import average_precision_score, classification_report

y_test = df_test['binary_class_id'].values
p_test = model.predict(test_classification_ds, verbose=0)[:, 0]
pred_test = (p_test >= DECISION_THRESHOLD).astype(int)

logging.info(f"Résultats de classification sur le jeu de test (seuil = {DECISION_THRESHOLD:.3f}) :")
logging.info(classification_report(y_test, pred_test, target_names=CLASS_NAMES, digits=4))
logging.info(f"PR-AUC debris (indépendante du seuil) : {average_precision_score(y_test, p_test):.4f}")

In [ ]:
# from pathlib import Path
# from tensorflow import keras

# MODEL_DIR = Path("models")
# model_paths = sorted(MODEL_DIR.glob("*.keras"))

# if not model_paths:
#     raise FileNotFoundError(f"Aucun modèle .keras trouvé dans {MODEL_DIR.resolve()}")

# validation_results = []

# for model_path in model_paths:
#     logging.info(f"Évaluation de {model_path.name}...")
#     candidate_model = keras.models.load_model(model_path)
#     balanced_metrics = candidate_model.evaluate(
#         val_classification_ds,
#         return_dict=True,
#         verbose=0,
#     )
#     original_metrics = candidate_model.evaluate(
#         val_original_classification_ds,
#         return_dict=True,
#         verbose=0,
#     )
#     validation_results.append({
#         "model": model_path.name,
#         "balanced_accuracy": float(balanced_metrics["accuracy"]),
#         "balanced_loss": float(balanced_metrics["loss"]),
#         "original_accuracy": float(original_metrics["accuracy"]),
#         "original_loss": float(original_metrics["loss"]),
#     })

# validation_results.sort(key=lambda result: result["balanced_accuracy"], reverse=True)

# logging.info("\nClassement des modèles sur les données de validation :")
# for rank, result in enumerate(validation_results, start=1):
#     logging.info(
#         f"{rank}. {result['model']} | "
#         f"équilibrée accuracy={result['balanced_accuracy']:.4f} | "
#         f"équilibrée loss={result['balanced_loss']:.4f} | "
#         f"originale accuracy={result['original_accuracy']:.4f} | "
#         f"originale loss={result['original_loss']:.4f}"
#     )

# validation_ranking = pd.DataFrame(validation_results)
# validation_ranking


In [ ]:
metrics = history.history

figure, axes = plt.subplots(1, 3, figsize=(18, 5))

for axis, (metric_name, title) in zip(axes, [
    ("loss", "Loss (pondérée par classe)"),
    ("pr_auc", "PR-AUC debris"),
    ("debris_recall", "Recall debris"),
]):
    axis.plot(history.epoch, metrics[metric_name], "o-", label="train")
    axis.plot(history.epoch, metrics[f"val_{metric_name}"], "o-", label="validation")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


try :
    figure, axes = plt.subplots(1, 2, figsize=(12, 5))

    for axis, (split_name, y_true, probabilities) in zip(axes, [
        ("validation", y_val, p_val),
        ("test", y_test, p_test),
    ]):
        y_pred = (probabilities >= DECISION_THRESHOLD).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(cmap="Blues", ax=axis, colorbar=False)
        axis.set_title(f"Matrice de confusion - {split_name} (seuil = {DECISION_THRESHOLD:.2f})")
    plt.tight_layout()
    plt.show()
except Exception as e:
    logging.error(f"Erreur lors de l'affichage des matrices de confusion : {e}", exc_info=True)

## 11. Comparaison de tous les modèles de classification

Cette cellule évalue chaque modèle enregistré dans `models` sur le jeu de test.

In [ ]:
from pathlib import Path

from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score

model_paths = sorted(Path("models").glob("*.keras"))
if not model_paths:
    raise FileNotFoundError("Aucun modèle .keras trouvé dans le dossier models.")

y_test_all_models = df_test["binary_class_id"].to_numpy(dtype=int)
comparison_results = []

for model_path in model_paths:
    try:
        candidate_model = keras.models.load_model(model_path, compile=False)
        probabilities = np.asarray(candidate_model.predict(test_classification_ds, verbose=0)).reshape(-1)
        predictions = (probabilities >= DECISION_THRESHOLD).astype(int)
        loss = float(np.mean(keras.losses.binary_crossentropy(y_test_all_models.astype("float32"), probabilities)))
        comparison_results.append({
            "modèle": model_path.name,
            "statut": "OK",
            "loss": loss,
            "accuracy": accuracy_score(y_test_all_models, predictions),
            "precision_debris": precision_score(y_test_all_models, predictions, zero_division=0),
            "recall_debris": recall_score(y_test_all_models, predictions, zero_division=0),
            "f1_debris": f1_score(y_test_all_models, predictions, zero_division=0),
            "pr_auc_debris": average_precision_score(y_test_all_models, probabilities),
        })
    except Exception as error:
        comparison_results.append({"modèle": model_path.name, "statut": f"ERREUR: {type(error).__name__}: {error}"})

classification_comparison = pd.DataFrame(comparison_results)
display(classification_comparison)